# Polyp-PVT Inference — Probability Maps

Generates probability maps (after sigmoid) for the Polyp-PVT test dataset.

- **ID**: Kvasir → `medical-imaging/data/polyp.npz`
- **OOD**: CVC-300, CVC-ClinicDB, CVC-ColonDB, ETIS-LaribPolypDB → `medical-imaging/data/polyp_ood.npz`

Each file contains:
- `p_hat`: probability array shape `(N, 1, 352, 352)` — output **after** sigmoid, values in `[0, 1]`
- `y`: binary masks shape `(N, 1, 352, 352)` — values in `{0, 1}`

> **Kernel**: use the `polyp_project` environment (created by notebook `00_register_polyp_project_kernel.ipynb`).

##### This code below is dedicated in case the probability maps are no longer available in the Polyp-PVT repository

In [1]:
import sys
import os

# Project root (where this notebook is located)
PROJECT_ROOT = os.path.abspath(os.getcwd())
POLYP_PVT_DIR = os.path.join(PROJECT_ROOT, 'Polyp-PVT-main')

# The model loads the backbone via relative path './pretrained_pth/pvt_v2_b2.pth',
# so we change the cwd to inside the repository before instantiating it.
os.chdir(POLYP_PVT_DIR)
sys.path.insert(0, POLYP_PVT_DIR)

import torch
import torch.nn.functional as F
import numpy as np
from torchvision import transforms
from PIL import Image
from tqdm import tqdm

from lib.pvt import PolypPVT

print('PyTorch version :', torch.__version__)
print('CUDA available   :', torch.cuda.is_available())
print('CWD              :', os.getcwd())

/home/bruno-borges/miniconda3/envs/polyp_project/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version : 2.5.1+cu121
CUDA available   : True
CWD              : /home/bruno-borges/soft_dice_confidence/tasks_models/polyp/Polyp-PVT-main


In [2]:
# ── Configuration ──────────────────────────────────────────────────────────────
TEST_SIZE   = 352
PTH_PATH    = os.path.join(POLYP_PVT_DIR, 'model_pth', 'PolypPVT .pth')
DATASET_DIR = os.path.join(PROJECT_ROOT, 'datasets', 'Polyp', 'TestDataset')
OUTPUT_DIR  = os.path.join(PROJECT_ROOT, 'inference_output', 'PolypPVT')

MEDICAL_DATA_DIR = os.path.normpath(os.path.join(PROJECT_ROOT, '..', '..', 'medical-imaging', 'data'))
os.makedirs(MEDICAL_DATA_DIR, exist_ok=True)

ID_DATASETS  = ['Kvasir']
OOD_DATASETS = ['CVC-300', 'CVC-ClinicDB', 'CVC-ColonDB', 'ETIS-LaribPolypDB']

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs(os.path.join(OUTPUT_DIR, 'ID'),  exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'OOD'), exist_ok=True)

print('Modelo   :', PTH_PATH)
print('Datasets :', DATASET_DIR)
print('Output   :', OUTPUT_DIR)
print('Device   :', DEVICE)

Modelo   : /home/bruno-borges/soft_dice_confidence/tasks_models/polyp/Polyp-PVT-main/model_pth/PolypPVT .pth
Datasets : /home/bruno-borges/soft_dice_confidence/tasks_models/polyp/datasets/Polyp/TestDataset
Output   : /home/bruno-borges/soft_dice_confidence/tasks_models/polyp/inference_output/PolypPVT
Device   : cuda


In [3]:
# ── Load the model ───────────────────────────────────────────────────────────
model = PolypPVT()
model.load_state_dict(torch.load(PTH_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()
print('Model loaded successfully.')

/home/bruno-borges/soft_dice_confidence/tasks_models/polyp/Polyp-PVT-main/lib/pvt.py:164: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  save_model = torch.load(path)
/tmp/ip

Model loaded successfully.


In [4]:
# ── Transforms ────────────────────────────────────────────────────────────────
img_transform = transforms.Compose([
    transforms.Resize((TEST_SIZE, TEST_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

gt_transform = transforms.Compose([
    transforms.Resize((TEST_SIZE, TEST_SIZE), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.ToTensor(),
])

def load_dataset_files(dataset_name):
    """Returns sorted lists of image and mask paths."""
    img_dir = os.path.join(DATASET_DIR, dataset_name, 'images')
    gt_dir  = os.path.join(DATASET_DIR, dataset_name, 'masks')
    imgs = sorted([os.path.join(img_dir, f) for f in os.listdir(img_dir)
                   if f.lower().endswith(('.jpg', '.png'))])
    gts  = sorted([os.path.join(gt_dir,  f) for f in os.listdir(gt_dir)
                   if f.lower().endswith(('.png', '.tif'))])
    assert len(imgs) == len(gts), (
        f'{dataset_name}: #imgs={len(imgs)} ≠ #masks={len(gts)}')
    return imgs, gts

In [5]:
# ── Inference function ───────────────────────────────────────────────────────
@torch.no_grad()
def run_inference(dataset_names):
    """
    Runs inference on the listed datasets and returns:
      p_hat — probabilities after sigmoid, shape (N,1,352,352), values in [0,1]
      y     — binary masks {0,1},            shape (N,1,352,352)
    """
    all_probs = []
    all_masks = []

    for ds_name in dataset_names:
        img_paths, gt_paths = load_dataset_files(ds_name)
        print(f'  {ds_name}: {len(img_paths)} images')

        for img_path, gt_path in tqdm(zip(img_paths, gt_paths),
                                      total=len(img_paths),
                                      desc=f'    {ds_name}',
                                      leave=True):
            # --- image ---
            with open(img_path, 'rb') as f:
                image = Image.open(f).convert('RGB')
            image_t = img_transform(image).unsqueeze(0).to(DEVICE)  # (1,3,352,352)

            # --- mask ---
            with open(gt_path, 'rb') as f:
                gt = Image.open(f).convert('L')
            gt_t = gt_transform(gt).unsqueeze(0)  # (1,1,352,352)

            # --- forward: logits → upsample → sigmoid → probabilities ---
            P1, P2 = model(image_t)                                  # outputs at ~88×88
            logits = F.interpolate(P1 + P2,
                                   size=(TEST_SIZE, TEST_SIZE),
                                   mode='bilinear',
                                   align_corners=False)               # (1,1,352,352)
            probs = torch.sigmoid(logits)                             # (1,1,352,352) ∈ [0,1]

            # --- binarize mask ---
            mask_bin = (gt_t > 0.5).float()                          # (1,1,352,352)

            all_probs.append(probs.cpu().numpy())     # float32 ∈ [0,1]
            all_masks.append(mask_bin.cpu().numpy())  # float32 {0,1}

    p_hat = np.concatenate(all_probs, axis=0)  # (N,1,352,352)
    y     = np.concatenate(all_masks, axis=0)  # (N,1,352,352)

    assert p_hat.shape == y.shape
    assert p_hat.shape[1] == 1
    assert p_hat.shape[2] == TEST_SIZE and p_hat.shape[3] == TEST_SIZE, \
        f'Unexpected shape: {p_hat.shape}'
    assert p_hat.min() >= 0.0 and p_hat.max() <= 1.0, \
        f'p_hat outside range [0,1]: [{p_hat.min():.4f}, {p_hat.max():.4f}]'

    return p_hat, y

In [6]:
# ── ID Inference (Kvasir) ─────────────────────────────────────────────────────
print('=== ID — Kvasir ===')
p_hat_id, y_id = run_inference(ID_DATASETS)

print(f'\np_hat shape : {p_hat_id.shape}')
print(f'y     shape : {y_id.shape}')
print(f'p_hat range : [{p_hat_id.min():.3f}, {p_hat_id.max():.3f}]  (expected: [0, 1])')
print(f'y unique    : {np.unique(y_id)}')

id_save = os.path.join(MEDICAL_DATA_DIR, 'polyp.npz')
np.savez_compressed(id_save, p_hat=p_hat_id, y=y_id)
print(f'\nSaved to: {id_save}')

=== ID — Kvasir ===
  Kvasir: 100 images


    Kvasir:   0%|          | 0/100 [00:00<?, ?it/s]/home/bruno-borges/soft_dice_confidence/tasks_models/polyp/Polyp-PVT-main/lib/pvt.py:91: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  edge = F.upsample(edge, (x.size()[-2], x.size()[-1]))
    Kvasir: 100%|██████████| 100/100 [00:02<00:00, 44.49it/s]



p_hat shape : (100, 1, 352, 352)
y     shape : (100, 1, 352, 352)
p_hat range : [0.000, 1.000]  (expected: [0, 1])
y unique    : [0. 1.]

Saved to: /home/bruno-borges/soft_dice_confidence/tasks_models/polyp/inference_output/PolypPVT/ID/data.npz


In [7]:
# ── OOD Inference (remaining datasets) ──────────────────────────────────────────
print('=== OOD — CVC-300 / CVC-ClinicDB / CVC-ColonDB / ETIS-LaribPolypDB ===')
p_hat_ood, y_ood = run_inference(OOD_DATASETS)

print(f'\np_hat shape : {p_hat_ood.shape}')
print(f'y     shape : {y_ood.shape}')
print(f'p_hat range : [{p_hat_ood.min():.3f}, {p_hat_ood.max():.3f}]  (expected: [0, 1])')
print(f'y unique    : {np.unique(y_ood)}')

ood_save = os.path.join(MEDICAL_DATA_DIR, 'polyp_ood.npz')
np.savez_compressed(ood_save, p_hat=p_hat_ood, y=y_ood)
print(f'\nSaved to: {ood_save}')

=== OOD — CVC-300 / CVC-ClinicDB / CVC-ColonDB / ETIS-LaribPolypDB ===
  CVC-300: 60 images


    CVC-300: 100%|██████████| 60/60 [00:01<00:00, 50.71it/s]


  CVC-ClinicDB: 62 images


    CVC-ClinicDB: 100%|██████████| 62/62 [00:00<00:00, 69.31it/s]


  CVC-ColonDB: 380 images


    CVC-ColonDB: 100%|██████████| 380/380 [00:07<00:00, 54.24it/s]


  ETIS-LaribPolypDB: 196 images


    ETIS-LaribPolypDB: 100%|██████████| 196/196 [00:08<00:00, 24.41it/s]



p_hat shape : (698, 1, 352, 352)
y     shape : (698, 1, 352, 352)
p_hat range : [0.000, 1.000]  (expected: [0, 1])
y unique    : [0. 1.]

Saved to: /home/bruno-borges/soft_dice_confidence/tasks_models/polyp/inference_output/PolypPVT/OOD/data.npz


In [8]:
# ── Final verification ──────────────────────────────────────────────────────────
print('=== Verification of generated files ===\n')

for split, path in [('ID', id_save), ('OOD', ood_save)]:
    data = np.load(path)
    ph = data['p_hat']
    y  = data['y']
    size_mb = os.path.getsize(path) / 1e6

    print(f'[{split}]  {path}  ({size_mb:.1f} MB)')
    print(f'  p_hat : shape={ph.shape}, dtype={ph.dtype}, '
          f'range=[{ph.min():.3f}, {ph.max():.3f}]')
    print(f'  y     : shape={y.shape},  dtype={y.dtype},  '
          f'unique={np.unique(y)}')

    assert ph.shape == y.shape,       'ERROR: shapes of p_hat and y differ'
    assert ph.shape[1] == 1,          'ERROR: channel != 1'
    assert ph.shape[2] == TEST_SIZE,  'ERROR: H != 352'
    assert ph.shape[3] == TEST_SIZE,  'ERROR: W != 352'
    assert ph.min() >= 0.0,           'ERROR: p_hat contains values < 0 (expected after sigmoid)'
    assert ph.max() <= 1.0,           'ERROR: p_hat contains values > 1 (expected after sigmoid)'
    print('  OK\n')

print('Inference completed successfully!')

=== Verification of generated files ===

[ID]  /home/bruno-borges/soft_dice_confidence/tasks_models/polyp/inference_output/PolypPVT/ID/data.npz  (37.9 MB)
  p_hat : shape=(100, 1, 352, 352), dtype=float32, range=[0.000, 1.000]
  y     : shape=(100, 1, 352, 352),  dtype=float32,  unique=[0. 1.]
  OK

[OOD]  /home/bruno-borges/soft_dice_confidence/tasks_models/polyp/inference_output/PolypPVT/OOD/data.npz  (287.1 MB)
  p_hat : shape=(698, 1, 352, 352), dtype=float32, range=[0.000, 1.000]
  y     : shape=(698, 1, 352, 352),  dtype=float32,  unique=[0. 1.]
  OK

Inference completed successfully!
